# Segmentación automática del surco de microirradiación láser

**La pregunta del trabajo.** Si merece la pena embeber un modelo de la familia SAM en un
plugin de Fiji para segmentar automáticamente la ROI del surco, o si un método clásico llega
lo bastante cerca como para no justificarlo. Y, dentro de SAM, qué variante: la hipótesis es
que las adaptadas a microscopía ganan a las adaptadas a radiología, y estas al linaje
original.

**Las decisiones de diseño están en la memoria del trabajo**, que manda sobre el código.
Este notebook es el índice ejecutable del proyecto: dice qué se ejecuta, en qué orden y con
qué entorno. **No calcula nada** salvo la comprobación final.

El código vive en `src/surco/`; los notebooks orquestan y muestran. Los pesos van fuera del
repositorio, en `SURCO_PESOS`, y los repositorios clonados de terceros en `SURCO_REPOS`.
`salidas/` no se versiona: se regenera entera con lo que hay aquí.

## Resumen: las diez fases y el apartado 7, de un vistazo

| fase | qué hace | entorno | notebook |
|---|---|---|---|
| **F1** | enumera unidades, fija el `start` y mide cada par | `surco` | `F1_inventario.ipynb` |
| **F2** | mide si el clic fijo se sale de la lesión | `surco` | `F2_deriva.ipynb` |
| **F3** | la única entrada humana del trabajo | `surco` | ninguno |
| **F4** | la maquinaria de medida, sin resultados propios | `surco` | cubierto por `tests/test_metricas.py` |
| **F5-F6** | un entorno conda por modelo, con su adaptador | **cada entorno de modelo** | ninguno |
| **F7** | los once modelos con la misma configuración fija | **cada entorno de modelo** | `F7_cribado.ipynb` |
| **F8** | la rejilla completa sobre los cuatro elegidos | **los cuatro entornos de la rejilla** | `F8_profundidad.ipynb` |
| **F9** | la hipótesis nula: tres métodos sin aprendizaje | `surco` | `F9_clasicos.ipynb` |
| **F10** | S0 contra S1 sobre las candidatas guardadas | `surco` | `F10_selector.ipynb` |
| **A7** | el techo humano, medido entre dos personas | `surco` | `A7_concordancia.ipynb` |

**Son diez fases, F1 a F10, y el apartado 7 encima**, que no es una fase del plan de
construcción sino una medida propia; la tabla lo lleva como `A7` para tenerlo todo a la
vista. La base cuenta igual (apartado 11).

Las fases van en orden: cada una consume lo que dejó la anterior. Las que necesitan un
entorno distinto de `surco` son **F5-F6, F7 y F8**, que es lo que dice la columna de arriba:
las dos primeras para construirlos y comprobar que todos los modelos cargan, y las otras dos
para medir con ellos. Son también las únicas que requieren GPU y pesos descargados.

## Lo que se reproduce solo con `surco`

**Seis de las diez fases, enteras (F1, F2, F3, F4, F9 y F10) y el apartado 7.** Sin GPU, sin
descargar un solo peso y sin más entorno que el del banco.

De las cuatro que faltan, **F7 y F8 dejan aquí sus agregados**, las tablas y los contrastes,
aunque la medida se haga dentro de los entornos de modelo; **F5 y F6 son los entornos**, así
que de esas no hay mitad que corra aquí. Los siete notebooks sí corren todos en `surco`.

Que F10 esté en esta lista no es casualidad y conviene entenderlo: F8 guardó **todas las
candidatas** de cada prompt empaquetadas a bits, y reordenar candidatas ya calculadas es
aritmética. Por eso el factor selector se mide entero sin volver a tocar un modelo.

**Con una condición:** esas candidatas están en `salidas/proceso/candidatas/` y **no se
versionan**, como el resto de `salidas/`. Quien clone el proyecto no las tiene, y hasta que
no corra F8 en los cuatro entornos de la rejilla, F10 no puede ejecutarse.

```bash
conda env create -f environment.yml
conda run -n surco pip install -e .
conda run -n surco pytest
```

## Lo que necesita los entornos por modelo

**F5-F6, F7 y F8.** Construirlos y comprobar que todos los modelos cargan es F5 y F6; medir
con ellos es F7 y F8. Cada modelo necesita versiones de torch y de sus repositorios que no
conviven entre sí, así que hay **un entorno conda por modelo** y el entorno del banco no
tiene torch ni ningún modelo instalado.

Diez entornos para el cribado y cuatro de ellos para la profundidad. Las recetas están en
`entornos/*.yml`.

```bash
conda env create -f entornos/<modelo>.yml
conda run -n surco-<modelo> python scripts/comprobar_carga.py
```

`scripts/entornos/` contiene además los parches a paquetes de terceros y las verificaciones
de exclusión. **No son opcionales**: son la evidencia reproducible de que se tocó código
ajeno, y sin ellos los entornos afectados no se pueden rehacer igual. Son **tres** los
entornos con parche (`edgesam`, `efficientvitsam` y `sam3`), y el tercero es el que hace
importable el paquete de SAM 3 en Windows, sin el cual su exclusión no se puede repetir.

## El orden completo, desde cero

Los pasos marcados con `<modelo>` se repiten una vez por entorno.

```bash
# entorno del banco
conda env create -f environment.yml
conda run -n surco pip install -e .
conda run -n surco pytest

# F1 y F2: los notebooks, en orden
# F3: anotacion (manual) y despues
conda run -n surco python scripts/verificar_clics.py
conda run -n surco python scripts/concordancia_seleccion.py

# F5 y F6: un entorno conda por modelo, y comprobar que todos cargan.
# Tres entornos llevan parche y se aplica antes de nada: ver entornos/LEEME.md
conda env create -f entornos/<modelo>.yml
conda run -n surco-<modelo> python scripts/comprobar_carga.py
conda run -n surco-<modelo> python scripts/entornos/verificar_prompt_mascara.py
# y una vez en cada uno de los dos excluidos, que es lo que hace repetible la exclusion
conda run -n surco-sam3    python scripts/entornos/verificar_exclusion_sam3.py
conda run -n surco-fastsam python scripts/entornos/verificar_exclusion_fastsam.py

# F7 y F8: lo unico que necesita los entornos por modelo para medir
conda run -n surco-<modelo> python scripts/cribado.py --todos-los-checkpoints
conda run -n surco python scripts/tabla_cribado.py
conda run -n surco-<modelo> python scripts/profundidad.py
conda run -n surco-<modelo> python scripts/m4_encadenado.py
conda run -n surco-medsam2 python scripts/medsam2_suelto.py
conda run -n surco python scripts/tabla_profundidad.py
conda run -n surco python scripts/memoria_por_dominio.py

# F9, F10 y apartado 7: todo en el banco
conda run -n surco python scripts/correr_clasicos.py
conda run -n surco python scripts/robustez_sigma.py
conda run -n surco python scripts/selector_s0_s1.py
conda run -n surco python scripts/selector_clasico.py
conda run -n surco python scripts/diagnostico_z.py
conda run -n surco python scripts/clasicos_contra_sam.py
conda run -n surco python scripts/tabla_selector.py
# `tabla_m4.py` es un agregado de F8 y aun asi corre aqui: su M1 de referencia
# en S1 sale de `selector.csv`, que lo escribe `selector_s0_s1.py` mas arriba.
conda run -n surco python scripts/tabla_m4.py
conda run -n surco python scripts/concordancia_anotadora.py --anotadora anotadora_1
conda run -n surco python scripts/concordancia_anotadora.py --anotadora anotadora_2
conda run -n surco python scripts/concordancia_entre_anotadoras.py

# las dos tablas finales: van las ultimas porque reunen todo lo anterior
conda run -n surco python scripts/contrastes.py
conda run -n surco python scripts/resumen.py
```

Los notebooks de resultados se ejecutan al final: `F7`, `F8`, `F9`, `F10` y `A7`.

Faltan de esta lista, a propósito, los tres `parchear_*.py` y los dos `verificar_<modelo>.py`:
se ejecutan al construir un entorno, no al reproducir resultados, y su sitio es
`entornos/LEEME.md`. Los dos de `scripts/unicos/` tampoco están, porque no se vuelven a
ejecutar (`README.md`).

## F1 · Inventario del ground truth

**Qué pregunta responde.** Cuántas lesiones hay, dónde empieza cada una y cómo es su geometría. Es el eslabón del que come todo lo demás: la unidad estadística del trabajo, la pareja `(serie, etiqueta)`, se define aquí. Su condición de salida es reproducir los tres recuentos declarados en el config: **17 unidades, 163 frames anotados y 312 pares**.

**Qué hay que ejecutar**, en `surco`:

```bash
# se ejecuta el propio notebook, no hay script
conda run -n surco jupyter lab notebooks/F1_inventario.ipynb
```

**Qué produce**, en `salidas/proceso/`: `inventario_gt.csv` (una fila por par unidad-frame) y `unidades.csv` (una por unidad, con su `start`).

**Dónde se ve:** `F1_inventario.ipynb`

## F2 · La medida previa del apartado 8

**Qué pregunta responde.** Si el clic puesto en el frame semilla sigue dentro de la lesión al avanzar los frames. Es lo que decide si el modo M2, corregir el clic por la deriva, hace falta. **Se sale en 183 de los 295 pares y en 15 de las 17 unidades**, así que hace falta; el apartado 5.1 explica por qué aun así no se puede implementar.

**Qué hay que ejecutar**, en `surco`:

```bash
# se ejecuta el propio notebook, no hay script
conda run -n surco jupyter lab notebooks/F2_deriva.ipynb
```

**Qué produce**, en `salidas/proceso/`: `deriva.csv` (una fila por par de la ventana), `deriva_por_unidad.csv` y `fragmentacion_en_start.csv`, que dice qué unidades ya llegan partidas al frame donde se anotan los clics.

**Dónde se ve:** `F2_deriva.ipynb`

## F3 · Anotación de clics y muestra de concordancia

**Qué pregunta responde.** Dónde clica un humano. Cinco clics por unidad sobre el frame semilla, con el protocolo del apartado 4.1, y se congelan en un fichero versionado. De aquí sale también la muestra de 34 pares estratificados que anotan las dos médicas. **Es el único paso que no es reproducible sin intervención manual**, y por eso su resultado se versiona en vez de regenerarse.

**Qué hay que ejecutar**, en `surco`:

```bash
conda run -n surco python -m surco.picker            # anotar (manual)
conda run -n surco python scripts/verificar_clics.py
conda run -n surco python scripts/concordancia_seleccion.py
```

**Qué produce:** `anotaciones/clics.json`, **versionado y fuera de `salidas/`** porque es la única entrada humana del trabajo y no se regenera; y en `salidas/`, `proceso/verificacion_clics.csv` y la carpeta `proceso/concordancia/` con las 21 imágenes y su tabla de correspondencia.

**Dónde se ve:** no tiene notebook.

## F4 · Métricas

**Qué pregunta responde.** Con qué se mide todo lo demás. Dice contra el GT, ventana de evaluación desde `start + 1`, agregación en dos pasos (dentro de cada lesión y después entre las 17), bootstrap remuestreando unidades y las dos curvas de umbrales. **No produce resultados**: produce la maquinaria, y se valida sobre casos sintéticos donde la respuesta correcta se conoce.

**Qué hay que ejecutar**, en `surco`:

```bash
conda run -n surco pytest tests/test_metricas.py -v
```

**Qué produce**, en `salidas/`: nada en `salidas/`.

**Dónde se ve:** cubierto por `tests/test_metricas.py`

## F5 y F6 · Los once modelos y sus entornos

**Qué pregunta responde.** Qué modelos se pueden ejecutar de verdad, y qué sabe hacer cada uno. Cada adaptador declara sus checkpoints, su lado de entrada, y si acepta clics negativos, prompt de máscara o banco de memoria. Dos modelos quedaron **excluidos con verificación reproducible**: SAM 3 y FastSAM.

**Qué hay que ejecutar**, en **cada entorno de modelo**:

```bash
conda env create -f entornos/<modelo>.yml
conda run -n surco-<modelo> python scripts/comprobar_carga.py
conda run -n surco-<modelo> python scripts/entornos/verificar_prompt_mascara.py
```

Y una vez en cada entorno de los dos excluidos, que es lo que hace reproducible la exclusión:

```bash
conda run -n surco-sam3    python scripts/entornos/verificar_exclusion_sam3.py
conda run -n surco-fastsam python scripts/entornos/verificar_exclusion_fastsam.py
```

**Qué produce**, en `salidas/proceso/`: `comprobacion_carga_surco-*.csv` y `prompt_mascara_surco-*.csv`, uno por entorno, más `exclusion_sam3.csv` y `exclusion_fastsam.csv`. Los cuatro **quedan fuera de la comprobación final** a propósito: solo existen si se levanta el entorno que los produce, así que anclarlos la haría fallar en cualquier copia normal. Lo que aportan es que los números del 5.1 y del 5.2 dejen de vivir solo en la consola.

**Dónde se ve:** no tiene notebook.

## F7 · Cribado

**Qué pregunta responde.** Cuáles de los once merecen pasar a la rejilla completa. Todos en **M1 × P1 × S0**, sobre las 17 unidades y en la misma ventana. Contesta además si el orden que sale sigue a la resolución de entrada o al dominio de ajuste, que es lo que hay que descartar antes de atribuir nada a la familia.

**Qué hay que ejecutar**, en **cada entorno de modelo**:

```bash
# una vez por cada uno de los diez entornos de modelo
conda run -n surco-<modelo> python scripts/cribado.py --todos-los-checkpoints
conda run -n surco python scripts/tabla_cribado.py
```

**Qué produce**, en `salidas/`: `cribado_surco-*.csv` y `coste_surco-*.csv` (uno por entorno), y los agregados `tabla_cribado.csv` y `tabla_coste.csv`.

**Dónde se ve:** `F7_cribado.ipynb`

## F8 · Profundidad

**Qué pregunta responde.** Si el tiempo aporta y cuántos clics necesita el biólogo. Cuatro modos por cuatro protocolos sobre los **k = 4** que salieron del cribado, uno por familia. Guarda **todas las candidatas empaquetadas a bits**, sin las cuales F10 no podría existir sin volver a ejecutar los modelos.

**Qué hay que ejecutar**, en **los cuatro entornos de la rejilla**:

```bash
# una vez por cada uno de los cuatro entornos de la rejilla
conda run -n surco-<modelo> python scripts/profundidad.py
conda run -n surco-<modelo> python scripts/m4_encadenado.py
conda run -n surco-medsam2 python scripts/medsam2_suelto.py
conda run -n surco python scripts/tabla_profundidad.py
conda run -n surco python scripts/memoria_por_dominio.py
```

**Qué produce**, en `salidas/proceso/`: `profundidad_surco-*.csv` y `m4_surco-*.csv`. Los dos últimos comandos **no guardan nada**: agregan e imprimen, y el notebook los recalcula al vuelo.

**Falta uno, y va después de F10 a propósito.** `scripts/tabla_m4.py` es el agregado de esta fase, pero compara M4 contra M1 **en los dos selectores**, y su M1 de referencia en S1 sale de `selector.csv`, que lo escribe F10. Corriéndolo aquí no encuentra ese fichero. Está en el orden completo de arriba, en el bloque de F10.

**Y en `salidas/proceso/candidatas/`**, los **153 ficheros `.npz`** con todas las candidatas de cada prompt, unos 62 MB. Van en su propia carpeta porque sueltos entre los CSV tapaban lo que hay que mirar. **Es la carpeta que hay que regenerar con esta fase si se clona el proyecto:** sin ella F10 no puede correr, y es lo único que obliga a volver a levantar los entornos de modelo para llegar a los resultados del selector.

**Dónde se ve:** `F8_profundidad.ipynb`

## F9 · Visión clásica

**Qué pregunta responde.** Si un método clásico llega lo bastante cerca como para que embeber SAM no esté justificado. Umbral local tipo Otsu, watershed con marcadores y filtro de cresta de Sato, los tres alimentados por los mismos cinco clics. **No depende de ningún entorno de modelo.**

**Qué hay que ejecutar**, en `surco`:

```bash
conda run -n surco python scripts/correr_clasicos.py
conda run -n surco python scripts/robustez_sigma.py
conda run -n surco python scripts/clasicos_contra_sam.py
```

**Qué produce**, en `salidas/`: `clasicos.csv`, `tabla_clasicos.csv` y `robustez_sigma.csv`. El último comando **no guarda nada**: agrega e imprime, y el notebook lo recalcula al vuelo.

**Dónde se ve:** `F9_clasicos.ipynb`

## F10 · El selector

**Qué pregunta responde.** Si el selector propio es aportación o adorno, y si sigue aportando cuando el prompt ya lleva clics de fondo. **No vuelve a ejecutar ningún modelo**: reordena las candidatas que dejó F8, que es aritmética. La excepción es M4, que se corre en F8 porque allí el selector cambia lo que el modelo ve en el frame siguiente.

**Qué hay que ejecutar**, en `surco`:

```bash
conda run -n surco python scripts/selector_s0_s1.py
conda run -n surco python scripts/selector_clasico.py
conda run -n surco python scripts/diagnostico_z.py
conda run -n surco python scripts/tabla_selector.py
conda run -n surco python scripts/tabla_m4.py
```

**Qué produce**, en `salidas/`: `selector.csv`, `selector_clasico.csv` y `diagnostico_z.csv`. Los dos últimos comandos **no guardan nada**: agregan e imprimen, y el notebook los recalcula al vuelo.

**El último es de F8 y corre aquí**, y conviene no confundirlo con lo anterior: lo que se mide en F8 es `m4_encadenado.py`, porque allí el selector cambia la cadena; lo que corre aquí es `tabla_m4.py`, que solo agrega. Va detrás porque compara M4 contra M1 en los dos selectores y su M1 en S1 sale de `selector.csv`, que lo escribe la primera línea de este bloque.

**Dónde se ve:** `F10_selector.ipynb`

## Apartado 7 · Concordancia entre anotadoras

**Qué pregunta responde.** Cuánto coinciden dos personas entrenadas que han visto la misma imagen y seguido el mismo protocolo. Da **el techo humano**, que es el único número del trabajo que no depende del autor, y permite tratar al modelo como un tercer anotador en vez de compararlo contra la referencia a la que está anclado.

**Qué hay que ejecutar**, en `surco`:

```bash
conda run -n surco python scripts/concordancia_anotadora.py --anotadora anotadora_1
conda run -n surco python scripts/concordancia_anotadora.py --anotadora anotadora_2
conda run -n surco python scripts/concordancia_entre_anotadoras.py
```

**Qué produce**, en `salidas/`: `concordancia_anotadora_{1,2}.csv`, `concordancia_a1_a2.csv`, `concordancia_tres_vias.csv` y `concordancia_tercer_anotador.csv`.

**Dónde se ve:** `A7_concordancia.ipynb`

## Las dos tablas finales

**Qué pregunta responden.** Ninguna nueva: reúnen lo que dejaron todas las fases para que citar un número no obligue a volver a ejecutar un notebook. Son dos y no una porque **hay dos formas de resultado que no caben en la misma tabla**: una fila por checkpoint medido y una fila por comparación pareada.

Van **las últimas** de todo el orden, y no por costumbre: `contrastes.py` lee las tres fuentes de la rejilla, el filtro clásico con y sin selector, el M3 suelto de MedSAM2 y las dos tablas del apartado 7, así que cualquiera que falte le deja filas fuera sin avisar.

**Qué hay que ejecutar**, en `surco`:

```bash
conda run -n surco python scripts/contrastes.py
conda run -n surco python scripts/resumen.py
```

**Qué produce**, en la raíz de `salidas/`, que es donde se queda solo lo que se mira:

- **`resumen.csv`**, una fila **por checkpoint medido**: familia, modelo, lado de entrada, Dice del cribado con su intervalo y el mismo resultado en IoU, si pasó a profundidad y su mejor celda, más el coste en dos columnas al final. **31 filas**, los 28 del cribado y los tres métodos clásicos. Lleva `LEEME_resumen.md` al lado explicando cada columna.
- **`contrastes.csv`**, una fila **por comparación pareada**: qué contra qué, diferencia, intervalo, unidades ganadas y si el intervalo se separa del cero. **232 filas** que cubren el factor modo, el factor prompt, el selector con sus dos mitades, las dos variantes del filtro clásico (el mismo selector, y trocear su respuesta en tres candidatas), el clásico contra cada modelo, **techo contra techo en las cuatro combinaciones**, la memoria contra el dominio, **los no elegidos del cribado contra los cuatro de la rejilla** y el techo humano. El nombre del clásico lleva la variante, `sato x P3 x una mascara`, porque hay tres números distintos detrás de `sato x P3`.

**Dónde se ve:** en los propios CSV. No tienen notebook: son la vista de conjunto que se comparte, no un análisis.

## ¿Está completa esta copia?

La celda de abajo recorre los CSV de `salidas/`, recalcula **un número ancla por fase** y lo
compara contra el valor que está escrito en la memoria.

Los valores esperados están copiados a mano desde la memoria **a propósito**: si se derivaran
de los mismos CSV que se quieren comprobar, la comprobación pasaría siempre y no diría nada.
Son la memoria mirando al código.

**Si alguna falla, hay dos causas posibles y las dos hay que mirarlas:** o falta por ejecutar
la fase que produce ese fichero, y el mensaje dirá qué CSV no encuentra, o el resultado se ha
movido respecto a lo que dice el texto, y entonces lo que hay que corregir es uno de los dos.

In [1]:
from surco import verificacion

comprobaciones = verificacion.comprobar()
print(verificacion.resumen(comprobaciones))
print()
print(comprobaciones.to_string(index=False))

Las 38 comprobaciones cuadran: la copia esta completa.

  apartado                                            que  memoria  salidas  cuadra
        F1                                       unidades  17.0000  17.0000    True
        F1                          pares (unidad, frame) 312.0000 312.0000    True
        F1                                frames anotados 163.0000 163.0000    True
        F2                        pares con el clic fuera 183.0000 183.0000    True
        F2                                  pares medidos 295.0000 295.0000    True
        F2                          unidades que se salen  15.0000  15.0000    True
        F7                                  filas medidas  28.0000  28.0000    True
        F7                             micro-sam LM vit_b   0.3389   0.3389    True
        F7                            SAM-Med2D sam_med2d   0.3284   0.3284    True
        F7                      Spearman Dice contra lado  -0.0390  -0.0390    True
        F8          